In [4]:
import pandas as pd
import numpy as np
import os

DATA_PATH = "../data/raw/"

ball = pd.read_csv(DATA_PATH + "ball_by_ball.csv")
matches = pd.read_csv(DATA_PATH + "matches.csv")
competitions = pd.read_csv(DATA_PATH + "competitions.csv")
teams = pd.read_csv(DATA_PATH + "teams.csv")
playing_xi = pd.read_csv(DATA_PATH + "playing_eleven.csv")
player_mapping = pd.read_csv(
    DATA_PATH + "competition_team_player_mapping.csv"
)

print("All datasets loaded successfully")

All datasets loaded successfully


In [6]:
datasets = {
    "Ball By Ball": ball,
    "Matches": matches,
    "Competitions": competitions,
    "Teams": teams,
    "Playing XI": playing_xi,
    "Player Mapping": player_mapping
}

for name, df in datasets.items():

    print("\nDataset:", name)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Duplicates:", df.duplicated().sum())
    print("Missing Values:", df.isnull().sum().sum())


Dataset: Ball By Ball
Rows: 200999
Columns: 33
Duplicates: 0
Missing Values: 766466

Dataset: Matches
Rows: 883
Columns: 22
Duplicates: 0
Missing Values: 2736

Dataset: Competitions
Rows: 141
Columns: 4
Duplicates: 0
Missing Values: 0

Dataset: Teams
Rows: 353
Columns: 3
Duplicates: 0
Missing Values: 0

Dataset: Playing XI
Rows: 28381
Columns: 9
Duplicates: 0
Missing Values: 26012

Dataset: Player Mapping
Rows: 6413
Columns: 6
Duplicates: 0
Missing Values: 0


In [7]:
for name, df in datasets.items():

    print("\n====================")
    print(name)
    print("====================")

    print(df.info())
    print(df.head())


Ball By Ball
<class 'pandas.DataFrame'>
RangeIndex: 200999 entries, 0 to 200998
Data columns (total 33 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   match_id             200999 non-null  int64  
 1   innings              200999 non-null  int64  
 2   overid               200999 non-null  int64  
 3   over_ball_num        200999 non-null  int64  
 4   innings_ball_num     200999 non-null  int64  
 5   batting_team_id      200999 non-null  int64  
 6   bowling_team_id      200999 non-null  int64  
 7   striker_id           200999 non-null  int64  
 8   striker_name         200999 non-null  str    
 9   non_striker_id       200999 non-null  int64  
 10  non_striker_name     200999 non-null  str    
 11  bowler_id            200999 non-null  int64  
 12  bowler_name          200999 non-null  str    
 13  batting_type         200999 non-null  str    
 14  bowling_type         200999 non-null  str    
 15  batting_order 

In [8]:
selected_competitions = competitions[
    (
        competitions["match_format"] == "T20I"
    ) |
    (
        competitions["comp_name"].str.contains(
            "IPL",
            case=False,
            na=False
        )
    )
]

print(
    selected_competitions[
        ["comp_id", "comp_name", "match_format", "season"]
    ].to_string(index=False)
)

 comp_id                                               comp_name match_format  season
    3395            South Africa in New Zealand T20I Series 2026         T20I    2026
    3405                Afghanistan v Sri Lanka T20I Series 2026         T20I    2026
    3465                      India in Zimbabwe T20I Series 2026         T20I    2026
    3568                 Bangladesh in Zimbabwe T20I Series 2026         T20I    2026
    3482                Australia in Bangladesh T20I Series 2026         T20I    2026
    3464               Sri Lanka in West Indies T20I Series 2026         T20I    2026
    3765                  Sri Lanka in Pakistan T20I Series 2026         T20I    2026
    3739                          India in Japan T20I Match 2026         T20I    2026
    3787                 Afghanistan v Zimbabwe T20I Series 2026         T20I    2026
    3713          Afghanistan vs India in India T20I Series 2026         T20I    2026
    3690                   India in New Zealand T20I S

In [9]:
valid_comp_ids = selected_competitions["comp_id"].unique()

selected_matches = matches[
    matches["comp_id"].isin(valid_comp_ids)
]

selected_match_ids = selected_matches["match_id"].unique()

selected_ball = ball[
    ball["match_id"].isin(selected_match_ids)
].copy()

print("Selected matches:", len(selected_matches))
print("Selected deliveries:", len(selected_ball))

Selected matches: 853
Selected deliveries: 194014


In [10]:
print(
    "Ball matches missing from matches:",
    len(set(ball.match_id) - set(matches.match_id))
)

print(
    "Matches missing ball data:",
    len(set(matches.match_id) - set(ball.match_id))
)

print(
    "Competition IDs missing:",
    len(set(matches.comp_id) - set(competitions.comp_id))
)

print(
    "Duplicate delivery keys:",
    selected_ball.duplicated(
        ["match_id", "innings", "innings_ball_num"]
    ).sum()
)

Ball matches missing from matches: 0
Matches missing ball data: 16
Competition IDs missing: 0
Duplicate delivery keys: 0
